<a href="https://colab.research.google.com/github/bushrahaider04/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bushrahaider04/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

1. My lane as an ML task (type)
Classification — I'm predicting whether a customer will churn in the next 30 days.

Why classification? The outcome is binary (will churn / will not churn), not a continuous value, cluster, or ranking. This is a straightforward yes/no prediction that maps naturally to classification algorithms like logistic regression, random forest, or XGBoost.

In [1]:
# Classification task demonstration
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Simulate customer churn data
np.random.seed(42)
n_customers = 1000

# Create features
data = {
    'customer_id': range(1001, 1001 + n_customers),
    'tenure_days': np.random.randint(1, 1000, n_customers),
    'monthly_usage_hours': np.random.exponential(50, n_customers),
    'support_tickets': np.random.poisson(2, n_customers),
    'avg_response_time_sec': np.random.exponential(120, n_customers),
    'total_spend': np.random.normal(50, 20, n_customers),
    'is_active': np.random.choice([0, 1], n_customers, p=[0.15, 0.85])
}

df = pd.DataFrame(data)
print("Dataframe preview:")
print(df.head())
print("\nDataframe shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)

Dataframe preview:
   customer_id  tenure_days  monthly_usage_hours  support_tickets  \
0         1001          103             2.313647                1   
1         1002          436             1.336041                3   
2         1003          861            23.617388                0   
3         1004          271            83.182386                2   
4         1005          107           218.213773                1   

   avg_response_time_sec  total_spend  is_active  
0             272.268632    41.797475          1  
1              26.222655    55.689513          0  
2               2.738758    70.702578          1  
3             237.369140    58.911292          1  
4             103.279191    87.179554          1  

Dataframe shape: (1000, 7)

Column types:
customer_id                int64
tenure_days                int64
monthly_usage_hours      float64
support_tickets            int64
avg_response_time_sec    float64
total_spend              float64
is_active          

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

2. Target or proxy
Target variable: churn_next_30_days — binary indicator (1 = will churn, 0 = will not churn)

Label source: This is a defined rule based on observed behavior — customers who have not logged in for 30 consecutive days will be labeled as churned. We're predicting future churn based on current patterns.

In [2]:
# Define the target based on a rule
# For demonstration, let's create a realistic target
def define_churn_rule(row):
    # A customer churns if:
    # 1. They have low tenure (under 100 days) AND
    # 2. Their usage is low (under 20 hours/month) AND
    # 3. They have more than 3 support tickets
    # This simulates a realistic business rule
    if (row['tenure_days'] < 100 and
        row['monthly_usage_hours'] < 20 and
        row['support_tickets'] > 3):
        return 1
    # Or if they haven't been active in the last 7 days (proxy for disengagement)
    elif row['is_active'] == 0:
        return 1
    else:
        return 0

df['churn_next_30_days'] = df.apply(define_churn_rule, axis=1)
print("Churn distribution:")
print(df['churn_next_30_days'].value_counts())
print(f"\nChurn rate: {df['churn_next_30_days'].mean():.2%}")

Churn distribution:
churn_next_30_days
0    845
1    155
Name: count, dtype: int64

Churn rate: 15.50%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: ROC-AUC (Area Under the Receiver Operating Characteristic Curve)

Why ROC-AUC?

Defensible: Works well with imbalanced data (typical in churn scenarios)

Interpretable: 0.5 = random, 1.0 = perfect

Business-friendly: Measures ranking quality — how well we separate churners from non-churners

What number means 'good'?

0.75+: Good performance — model has real predictive power

0.70+: Acceptable — useful for decision support

0.80+: Excellent — strong signal in the data

In [3]:
# Demonstrate ROC-AUC with a simple model
features = ['tenure_days', 'monthly_usage_hours', 'support_tickets',
            'avg_response_time_sec', 'total_spend']
X = df[features]
y = df['churn_next_30_days']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train simple model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict probabilities
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate ROC-AUC
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {auc_score:.3f}")

# Classification report for additional context
y_pred = model.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Evaluate if this is 'good'
if auc_score >= 0.75:
    print("\n✅ ROC-AUC >= 0.75 — This is GOOD performance!")
elif auc_score >= 0.70:
    print("\n⚠️ ROC-AUC between 0.70 and 0.75 — Acceptable, but could improve")
else:
    print("\n❌ ROC-AUC < 0.70 — Needs improvement for practical use")

ROC-AUC Score: 0.573

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.85      0.98      0.91       254
       Churn       0.43      0.07      0.11        46

    accuracy                           0.84       300
   macro avg       0.64      0.52      0.51       300
weighted avg       0.79      0.84      0.79       300


❌ ROC-AUC < 0.70 — Needs improvement for practical use


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

4. The unit of analysis, as a real dataframe
One row = one customer at a specific point in time

Each row represents an individual customer's features and whether they'll churn in the next 30 days. This is a customer-level analysis where we're predicting behavior for each unique customer.

In [4]:
# Show the dataframe with clear unit of analysis
print("Unit of Analysis: One row = One Customer")
print("=" * 50)
print("\nDataframe with 5 sample customers:")
display_df = df.head(10).copy()
display_df['customer_id'] = display_df['customer_id'].astype(str)
print(display_df[['customer_id', 'tenure_days', 'monthly_usage_hours',
                   'support_tickets', 'churn_next_30_days']].to_string(index=False))

print("\nDataframe details:")
print(f"- Total rows (customers): {len(df):,}")
print(f"- Total features: {len(df.columns)}")
print(f"- Target column: 'churn_next_30_days'")
print(f"- Time frame: Current features → Future churn (next 30 days)")

Unit of Analysis: One row = One Customer

Dataframe with 5 sample customers:
customer_id  tenure_days  monthly_usage_hours  support_tickets  churn_next_30_days
       1001          103             2.313647                1                   0
       1002          436             1.336041                3                   1
       1003          861            23.617388                0                   0
       1004          271            83.182386                2                   0
       1005          107           218.213773                1                   0
       1006           72             8.150475                1                   0
       1007          701            45.086207                2                   0
       1008           21            23.973685                2                   0
       1009          615           175.185428                1                   0
       1010          122            92.295660                1                   0

Dataframe

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pattern is too messy because:

Complex interactions: Churn depends on the combination of multiple factors (tenure × usage × support tickets × engagement) — not just a single threshold.

Non-linear relationships: The relationship between usage and churn isn't linear. Low usage predicts churn, but very high usage might also indicate power users who are at risk for different reasons.

Changing patterns over time: Customer behavior evolves, and what predicted churn 6 months ago might not work today.

Individual variation: People churn for different reasons — some because of price, others because of poor support, others because they don't use the product enough.

In [5]:
# Demonstrate why ML beats a fixed rule
def fixed_rule_churn(row):
    """A simple rule-based approach"""
    if (row['tenure_days'] < 60 or
        row['monthly_usage_hours'] < 15 or
        row['support_tickets'] > 5):
        return 1
    return 0

# Compare ML vs fixed rule
df['fixed_rule_pred'] = df.apply(fixed_rule_churn, axis=1)

# Show where ML and fixed rule disagree
disagreement = df[df['fixed_rule_pred'] != df['churn_next_30_days']]
print(f"Disagreements between fixed rule and true churn: {len(disagreement):,} of {len(df):,} ({len(disagreement)/len(df):.1%})")

# Show examples of complex patterns
print("\nExample of complex pattern: Customer with long tenure but still churns")
complex_cases = df[(df['tenure_days'] > 200) & (df['churn_next_30_days'] == 1)].head(3)
if len(complex_cases) > 0:
    for _, row in complex_cases.iterrows():
        print(f"Customer {int(row['customer_id'])}: Tenure={row['tenure_days']} days, "
              f"Usage={row['monthly_usage_hours']:.1f} hrs, "
              f"Tickets={row['support_tickets']}, "
              f"Active={row['is_active']} → Churned (ML prediction: {row['churn_next_30_days']})")

print("\n🔍 Why ML is better:")
print("✓ Captures non-linear patterns and interactions")
print("✓ Adapts to changing customer behavior")
print("✓ Provides probability scores for risk ranking")
print("✓ Can handle missing data and noisy features")
print("✓ Identifies subtle signals that rules miss")

Disagreements between fixed rule and true churn: 362 of 1,000 (36.2%)

Example of complex pattern: Customer with long tenure but still churns
Customer 1002: Tenure=436.0 days, Usage=1.3 hrs, Tickets=3.0, Active=0.0 → Churned (ML prediction: 1.0)
Customer 1011: Tenure=467.0 days, Usage=91.1 hrs, Tickets=1.0, Active=0.0 → Churned (ML prediction: 1.0)
Customer 1030: Tenure=956.0 days, Usage=55.7 hrs, Tickets=0.0, Active=0.0 → Churned (ML prediction: 1.0)

🔍 Why ML is better:
✓ Captures non-linear patterns and interactions
✓ Adapts to changing customer behavior
✓ Provides probability scores for risk ranking
✓ Can handle missing data and noisy features
✓ Identifies subtle signals that rules miss


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.